In [1]:
# Standard library imports
import os
import sys
import random
import warnings
import math

# Third-party numerical and data handling
import numpy as np
import pandas as pd
import h5py
import cv2
from PIL import Image

# Visualization
import matplotlib.pyplot as plt
from tqdm import tqdm

# Machine learning utilities
from sklearn.metrics import (
    average_precision_score,
    label_ranking_average_precision_score,
    roc_auc_score
)

# PyTorch core
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler

# PyTorch vision
from torchvision import models

# Albumentations
import albumentations as A
from albumentations.core.transforms_interface import ImageOnlyTransform
from albumentations.pytorch import ToTensorV2

from losses import AsymmetricLossOptimized
from preprocess import ecg_processing_pipeline, smart_pad_and_resize_ecg, ecg_processing_pipeline_no_perspective_distortion
from transformations import CornerCutout, GradientShadow, PaperFoldEffect, BottomBlur

### Helpers

In [2]:
def check_device():
    """
    Check available compute devices and return the best one.
    Priority: CUDA > MPS > CPU
    """
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("✓ CUDA available")
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("✓ MPS (Apple Silicon GPU) available")
    else:
        device = torch.device("cpu")
        print("✗ Using CPU (no GPU acceleration available)")
    
    print(f"\nSelected device: {device}")
    return device

# Check and get device
device = check_device()

✓ MPS (Apple Silicon GPU) available

Selected device: mps


### Architecture

In [3]:
class Head(nn.Module):
    def __init__(self, in_features, hidden_layer, dropout_rate=0.3):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features, hidden_layer),
            nn.BatchNorm1d(hidden_layer),  
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_layer, hidden_layer // 2), 
            nn.BatchNorm1d(hidden_layer // 2),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_layer // 2, 1)
        )
    
    def forward(self, x):
        return self.layers(x)

class MultiHeadEfficientNet(nn.Module):
    def __init__(self, num_conditions=5, hidden_dim=512, dropout_rate=0.3):
        super().__init__()
        
        backbone = models.convnext_base(weights="IMAGENET1K_V1", progress=True)
        in_features = backbone.classifier[2].in_features
        assert isinstance(in_features, int), f"in_features should be int, got {type(in_features)}"
        
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        
        self.shared_feature_processor = nn.Sequential(
            nn.Linear(in_features, in_features), 
            nn.BatchNorm1d(in_features),
            nn.GELU(), 
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout_rate)
        )
        
        self.heads = nn.ModuleList([
            Head(hidden_dim, hidden_dim // 2, dropout_rate) 
            for _ in range(num_conditions)
        ])
        
    def forward(self, x):
        backbone_feats = self.backbone(x).flatten(1) # squeeze non-batch dimension
        processed_feats = self.shared_feature_processor(backbone_feats)
        outputs = [head(processed_feats) for head in self.heads]
        return torch.cat(outputs, dim=1)  # [batch, num_conditions]

### Image Preprocessing/DataLoader

In [4]:
root_path = os.path.abspath("../")
data_path = os.path.join(root_path, "data")

In [5]:
def load_contours_from_hdf5(filepath='/kaggle/input/ecg-image-contours/contours.h5'):
    """
    Load all contours from HDF5 file back into dictionary format
    """
    contour_dict = {}
    
    with h5py.File(filepath, 'r') as f:
        for img_id in f.keys():
            grp = f[img_id]
            
            contour_dict[img_id] = {
                'contour': grp['contour'][:],  # Load the contour array
                'scale_x': grp.attrs['scale_x'],
                'scale_y': grp.attrs['scale_y'], 
                'half': grp.attrs['half']
            }
    
    return contour_dict

# Usage
try: 
    print(contours["train_000000"])
except Exception as e: 
    contours = load_contours_from_hdf5(os.path.join(data_path, "contours.h5"))

In [6]:
import time as time

def process_single_img(
                img,
                img_id,
                desired_aspect=0.5,
                target_width=512
                ):
    value = f"train_{str(img_id).zfill(6)}.png"
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    # output = ecg_processing_pipeline(input_image = img, 
    #                             contour_data = contours[value.split(".")[0]])
    output = ecg_processing_pipeline_no_perspective_distortion(input_image = img, 
                                contour_data = contours[value.split(".")[0]])
    resize_output = smart_pad_and_resize_ecg(output, target_size=(int(target_width*desired_aspect), 512), resize_strategy=cv2.INTER_AREA)
    return resize_output

# def imread_clean(path):
#     img = Image.open(path)
#     img = img.convert('RGB')  # Strips metadata
#     return np.array(img)

def imread_clean(path):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        img = cv2.imread(path)
        if img is None:
            raise ValueError(f"Failed to load image: {path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img
    
class ECGDataset(Dataset): 
    def __init__(self,
                image_paths, 
                labels_df, 
                transforms=None):
        
        self.image_paths = image_paths
        self.labels_dict = {idx: torch.tensor(row.values, dtype=torch.float32) 
                            for idx, row in labels_df.iterrows()}
        
        self.idx_to_image_id = {}
        for idx, path in enumerate(self.image_paths):
            filename = os.path.basename(path)
            image_id = int(filename.rsplit('_', 1)[-1].split('.')[0])
            self.idx_to_image_id[idx] = image_id
        
        self.transforms = transforms
        
    def __len__(self): 
        return len(self.image_paths)
        
    def __getitem__(self, idx): 
        time0 = time.time()
        image_path = self.image_paths[idx]
        index = self.idx_to_image_id[idx]
        label = self.labels_dict[index]
        time1 = time.time()
        image = imread_clean(image_path)
        if image is None:
            raise ValueError(f"Failed to load image: {image_path}")
        time2 = time.time()
        # convert image using our segmentation pipeline 
        image_conv = process_single_img(
            img=image, 
            img_id=index,
            desired_aspect=0.5, 
            target_width=512
        )
        image_conv = np.stack([image_conv, image_conv, image_conv], axis=-1)
        time3 = time.time()
        if self.transforms is not None: 
            image_tens = self.transforms(image=image_conv)["image"]
        else: 
            image_tens = image_conv
        time4 = time.time()
        #print(f"Collect: {t1-t0:.3f}s, Load: {t2-t1:.3f}s, Process: {t3-t2:.3f}s, Aug: {t4-t3:.3f}s")
        return image_tens, label

In [7]:
train_transforms = A.Compose([
    A.CLAHE(
        clip_limit=4,
        tile_grid_size=(8, 8),
        p=1.0
    ),
    #Data Augmentations
    A.Rotate(limit=2, p=0.3),  # small rotations, limit is +/- degrees
    A.Affine(translate_percent={'x': (-0.1, 0.1), 'y': (-0.05, 0.05)}, 
            rotate=0, scale=1.0, shear=0, p=0.3),  # small translations
    A.RandomShadow( # shadows
        shadow_roi=(0, 0, 1, 1),  # Can appear anywhere in image
        num_shadows_limit=(1,2),  # 1-2 shadow regions
        shadow_dimension=4,         # Controls shadow size/complexity
        shadow_intensity_range=(0.2, 0.4),
        p=0.3
    ),
    A.ElasticTransform(
        alpha=30, 
        sigma=15, 
        interpolation=cv2.INTER_AREA,
        p=0.3
    ),
    A.GaussianBlur(
        blur_limit=0, 
        sigma_limit=(0.1, 1.0),
        p=0.3
    ),
    A.Perspective(
        scale=[0.01, 0.03],
        keep_size=True,
        fit_output=True,
        interpolation=cv2.INTER_AREA,
        mask_interpolation=cv2.INTER_AREA,
        border_mode=cv2.BORDER_CONSTANT,
        fill=0,
        fill_mask=0,
        p=0.3
    ),
    # Normalization (ImageNet)
    A.Normalize(mean=[0.485, 0.456, 0.406], 
                std=[0.229, 0.224, 0.225]),
    # Convert to tensor and replicate grayscale to 3 channels
    ToTensorV2()
])

val_transforms = A.Compose([
    A.CLAHE(
        clip_limit=4,
        tile_grid_size=(8, 8),
        p=1.0
    ),
    # Normalization (ImageNet)
    A.Normalize(mean=[0.485, 0.456, 0.406], 
                std=[0.229, 0.224, 0.225]),
    # Convert to tensor
    ToTensorV2()
])

In [8]:
from sklearn.model_selection import train_test_split

labels_df = pd.read_csv(os.path.join(data_path, "train_final.csv"), index_col=0, dtype=int)
image_path = os.path.join(root_path, "broken_images")
with open(os.path.join(image_path, "broken_images_list.txt"), "r") as file:
    broken_train_images = [line.strip() for line in file]
with open(os.path.join(image_path, "broken_test_images_list.txt"), "r") as file:
    broken_test_images = [line.strip() for line in file]
with open(os.path.join(image_path, "valid_images_list.txt"), "r") as file:
    valid_train_images = [line.strip() for line in file]
with open(os.path.join(image_path, "valid_test_images_list.txt"), "r") as file:
    valid_test_images = [line.strip() for line in file]

ids = set([int(obj.split(".")[-2][-6:]) for obj in valid_train_images])

labels_df = labels_df.loc[labels_df.index.isin(ids)]

# split train set, using stratification 
X_train, X_test, y_train, y_test = train_test_split(valid_train_images, 
                                                    labels_df, 
                                                    test_size = 0.2,
                                                    random_state = 42, 
                                                    shuffle = True, 
                                                    stratify = labels_df[["CD", "MI", "AF", "STTC", "HYP"]])


In [9]:
print(f"Full number of samples: Train = {len(X_train)}, Val = {len(X_test)}")

Full number of samples: Train = 12006, Val = 3002


In [10]:
# need to filter images so that we only try to use the ones that we have local copies of
# and also change the filenames

import glob 
local_files = glob.glob(os.path.abspath(os.path.join(data_path, "train", "train_*")))
indices = set([int(item.split(".")[0][-6:]) for item in local_files])

# usable local ids 
local_ids = indices.intersection(ids)

# downsample ids to be more tractable numbers
#local_ids = [id_val for ind, id_val in enumerate(local_ids) if ind%10 == 0]

# filter X_train and X_test to only contain the correct ids
X_train_usable = [elem for elem in X_train if int(elem.split(".")[0][-6:]) in local_ids]
X_test_usable = [elem for elem in X_test if int(elem.split(".")[0][-6:]) in local_ids]

# rewrite paths to actually be the local ones
X_train = [os.path.abspath(os.path.join(data_path, "train", path.split("/")[-1])) for path in X_train_usable]
X_test = [os.path.abspath(os.path.join(data_path, "train", path.split("/")[-1])) for path in X_test_usable]

print(f"Actual number of local samples: Train = {len(X_train)}, Val = {len(X_test)}")

Actual number of local samples: Train = 4019, Val = 980


In [13]:
num_workers = 0 if sys.platform == 'darwin' else 4 
print(f"Using num_workers = {num_workers}")

train_dataset = ECGDataset(
                    image_paths=X_train, 
                    labels_df=labels_df, 
                    transforms=train_transforms, 
                    )
val_dataset = ECGDataset(
                    image_paths=X_test, 
                    labels_df=labels_df, 
                    transforms=val_transforms, 
                    ) 

train_dataloader = DataLoader( 
                        train_dataset,
                        batch_size=8, 
                        shuffle=True, 
                        num_workers=num_workers, 
                        pin_memory=True if device.type == "cuda" else False)
val_dataloader = DataLoader( 
                        val_dataset,
                        batch_size=8, 
                        shuffle=False, 
                        num_workers=num_workers, 
                        pin_memory=True if device.type == "cuda" else False)

Using num_workers = 0


In [14]:
import time
import numpy as np

# Test on 10 different images
test_indices = [0, 100, 200, 300, 400, 500, 600, 700, 800, 900]

results = {
    'setup': [],
    'imread_clean': [],
    'process_single_img': [],
    'stack_channels': [],
    'transforms': [],
    'total': []
}

for idx in test_indices:
    t0 = time.time()
    image_path = train_dataset.image_paths[idx]
    index = train_dataset.idx_to_image_id[idx]
    label = train_dataset.labels_dict[index]
    t1 = time.time()
    
    image = imread_clean(image_path)
    t2 = time.time()
    
    image_conv = process_single_img(img=image, img_id=index, desired_aspect=0.5, target_width=512)
    t3 = time.time()
    
    image_conv = np.stack([image_conv, image_conv, image_conv], axis=-1)
    t4 = time.time()
    
    image_tens = train_dataset.transforms(image=image_conv)["image"]
    t5 = time.time()
    
    results['setup'].append(t1 - t0)
    results['imread_clean'].append(t2 - t1)
    results['process_single_img'].append(t3 - t2)
    results['stack_channels'].append(t4 - t3)
    results['transforms'].append(t5 - t4)
    results['total'].append(t5 - t0)
    
    print(f"Image {idx}: Total={t5-t0:.3f}s (Load={t2-t1:.3f}s, Process={t3-t2:.3f}s, Stack={t4-t3:.3f}s, Transform={t5-t4:.3f}s)")

print("\n" + "="*60)
print("AVERAGES:")
print("="*60)
for key, times in results.items():
    print(f"{key:20s}: {np.mean(times):.3f}s (±{np.std(times):.3f}s)")

print("\n" + "="*60)
print(f"Estimated batch time (batch_size=8): {np.mean(results['total']) * 8:.3f}s")
print("="*60)

Image 0: Total=0.129s (Load=0.114s, Process=0.012s, Stack=0.000s, Transform=0.002s)
Image 100: Total=0.111s (Load=0.096s, Process=0.013s, Stack=0.000s, Transform=0.002s)
Image 200: Total=0.113s (Load=0.097s, Process=0.013s, Stack=0.000s, Transform=0.002s)
Image 300: Total=0.113s (Load=0.100s, Process=0.011s, Stack=0.000s, Transform=0.003s)
Image 400: Total=0.120s (Load=0.103s, Process=0.015s, Stack=0.000s, Transform=0.003s)
Image 500: Total=0.121s (Load=0.100s, Process=0.015s, Stack=0.000s, Transform=0.007s)
Image 600: Total=0.110s (Load=0.096s, Process=0.012s, Stack=0.000s, Transform=0.002s)
Image 700: Total=0.121s (Load=0.097s, Process=0.018s, Stack=0.000s, Transform=0.006s)
Image 800: Total=0.112s (Load=0.097s, Process=0.013s, Stack=0.000s, Transform=0.002s)
Image 900: Total=0.107s (Load=0.097s, Process=0.008s, Stack=0.000s, Transform=0.002s)

AVERAGES:
setup               : 0.000s (±0.000s)
imread_clean        : 0.100s (±0.005s)
process_single_img  : 0.013s (±0.002s)
stack_channels

### Metrics for Evaluation

In [25]:
def batch_training_metrics(y_true, y_pred):
    """Compute sums that can be averaged later."""
    y_true = y_true.float()
    y_pred = y_pred.float()
    
    # Sum predicted probs per label
    sum_pred_prob_per_label = torch.sum(y_pred, dim=0)  # (L,)
    
    # Sum ground truth per label
    sum_true_per_label = torch.sum(y_true, dim=0)       # (L,)

    # Soft cardinality
    soft_cardinality_sum = torch.sum(torch.sum(y_pred, dim=1))  # scalar

    # Probability mass
    prob_mass_sum = torch.sum(y_pred)  # scalar
    
    # Calibration metrics - accumulate per label
    sum_pred_given_positive = torch.sum(y_pred * y_true, dim=0)  # (L,)
    sum_pred_given_negative = torch.sum(y_pred * (1 - y_true), dim=0)  # (L,)
    count_positive = torch.sum(y_true, dim=0)  # (L,)
    count_negative = torch.sum(1 - y_true, dim=0)  # (L,)

    return {
        "sum_pred_prob_per_label": sum_pred_prob_per_label,
        "sum_true_per_label": sum_true_per_label,
        "soft_cardinality_sum": soft_cardinality_sum,
        "prob_mass_sum": prob_mass_sum,
        "sum_pred_given_positive": sum_pred_given_positive,
        "sum_pred_given_negative": sum_pred_given_negative,
        "count_positive": count_positive,
        "count_negative": count_negative,
    }


def aggregate_training_epoch(batch_stats, total_samples):
    L = batch_stats[0]["sum_pred_prob_per_label"].shape[0]

    total_pred_prob = torch.zeros(L)
    total_true = torch.zeros(L)
    total_prob_mass = 0.0
    total_cardinality = 0.0
    total_pred_pos = torch.zeros(L)
    total_pred_neg = torch.zeros(L)
    total_count_pos = torch.zeros(L)
    total_count_neg = torch.zeros(L)

    for s in batch_stats:
        total_pred_prob += s["sum_pred_prob_per_label"].cpu()
        total_true += s["sum_true_per_label"].cpu()
        total_prob_mass += s["prob_mass_sum"].item()
        total_cardinality += s["soft_cardinality_sum"].item()
        total_pred_pos += s["sum_pred_given_positive"].cpu()
        total_pred_neg += s["sum_pred_given_negative"].cpu()
        total_count_pos += s["count_positive"].cpu()
        total_count_neg += s["count_negative"].cpu()
    
    mean_pred_when_positive = (total_pred_pos / (total_count_pos + 1e-8)).tolist()
    mean_pred_when_negative = (total_pred_neg / (total_count_neg + 1e-8)).tolist()

    return {
        "mean_pred_prob_per_label": (total_pred_prob / total_samples).tolist(),
        "mean_true_prob_per_label": (total_true / total_samples).tolist(),
        "mean_prob_mass": total_prob_mass / total_samples,
        "mean_cardinality": total_cardinality / total_samples,
        "mean_pred_when_positive": mean_pred_when_positive,
        "mean_pred_when_negative": mean_pred_when_negative,
        "calibration_gap": [(p - n) for p, n in zip(mean_pred_when_positive, mean_pred_when_negative)],
    }

def compute_ranking_metrics(all_y_true, all_y_pred):
    """Compute AP/AUROC/LRAP ranking metrics. Inputs are numpy arrays."""
    L = all_y_true.shape[1]

    per_label_ap = []
    per_label_auroc = []
    
    for j in range(L):
        # Average Precision
        ap = average_precision_score(all_y_true[:, j], all_y_pred[:, j])
        per_label_ap.append(float(ap))
        
        # AUROC
        try:
            auroc = roc_auc_score(all_y_true[:, j], all_y_pred[:, j])
            per_label_auroc.append(float(auroc))
        except ValueError:
            # Handle case where only one class is present in y_true
            per_label_auroc.append(float('nan'))

    # Micro-averaged metrics
    micro_ap = average_precision_score(all_y_true.reshape(-1), all_y_pred.reshape(-1))
    try:
        micro_auroc = roc_auc_score(all_y_true.reshape(-1), all_y_pred.reshape(-1))
    except ValueError:
        micro_auroc = float('nan')
    
    # Macro-averaged metrics
    macro_ap = sum(per_label_ap) / L
    valid_aurocs = [x for x in per_label_auroc if not np.isnan(x)]
    macro_auroc = sum(valid_aurocs) / len(valid_aurocs) if valid_aurocs else float('nan')
    
    # LRAP
    lrap = label_ranking_average_precision_score(all_y_true, all_y_pred)

    return {
        "per_label_ap": per_label_ap,
        "per_label_auroc": per_label_auroc,
        "macro_ap": float(macro_ap),
        "macro_auroc": float(macro_auroc),
        "micro_ap": float(micro_ap),
        "micro_auroc": float(micro_auroc),
        "lrap": float(lrap),
    }

### Initialise useful training functions

In [26]:
# load checkpoint 
checkpoint_path = "convnext_checkpoint.pth"
has_checkpoint = False
try: 
    checkpoint = torch.load(checkpoint_path, map_location=torch.device("cpu"), weights_only=False)
    has_checkpoint = True
    current_epoch = checkpoint["epoch"]
    print(f"Loading from checkpoint, last run epoch was {current_epoch}")
except Exception as e: 
    print("No checkpoint found, continuing as default")
    current_epoch = 0
    
num_epochs_decay = 80
num_epochs_frozen = 3
num_epochs_const = 10
num_warmup_epochs = 3
num_epochs_total = num_epochs_decay + num_epochs_frozen + num_epochs_const + num_warmup_epochs

No checkpoint found, continuing as default


In [27]:
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """
    Multi-label focal loss with optional per-class alpha and gamma.
    Handles device placement automatically and ensures numerical stability.
    """

    def __init__(self, gamma=2.0, alpha=None, reduction="mean"):
        """
        Args:
            gamma (float or tensor): focusing parameter; scalar or per-class vector.
            alpha (float or tensor): class balance weights; scalar or per-class vector.
            reduction (str): "mean", "sum", or "none".
        """
        super().__init__()
        self.reduction = reduction

        # ---- Store gamma (scalar or vector) ----
        if torch.is_tensor(gamma):
            self.register_buffer("gamma", gamma.float())
            self.gamma_is_scalar = False
        else:
            self.gamma = float(gamma)
            self.gamma_is_scalar = True

        # ---- Store alpha (scalar or vector) ----
        if alpha is not None:
            if not torch.is_tensor(alpha):
                alpha = torch.tensor(alpha, dtype=torch.float32)
            self.register_buffer("alpha", alpha.float())
            self.alpha_is_set = True
        else:
            self.alpha_is_set = False

    def forward(self, logits, targets):
        """
        Args:
            logits: raw model outputs (batch, num_classes)
            targets: binary labels (batch, num_classes)
        """

        # ---- Numerically stable sigmoid + BCE ----
        # Instead of sigmoid(logits) then BCE, we use the built-in stable function.
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, reduction="none"
        )

        # Stable sigmoid
        p = torch.sigmoid(logits)

        # p_t = p for y=1, else 1-p
        pt = p * targets + (1 - p) * (1 - targets)

        # ---- Make sure gamma and alpha match device and shape ----
        if self.gamma_is_scalar:
            gamma = self.gamma
        else:
            gamma = self.gamma.to(logits.device)  # (num_classes,)

        if self.alpha_is_set:
            alpha = self.alpha.to(logits.device)  # (num_classes,)
            alpha_t = alpha * targets + 1.0 * (1 - targets)
        else:
            alpha_t = 1.0

        # ---- Focal modulation ----
        # Add eps for numerical stability: (1 − pt) never becomes exactly 0
        eps = 1e-8
        focal_weight = (1 - pt + eps) ** gamma

        # ---- Apply alpha weighting ----
        focal_weight = focal_weight * alpha_t

        # ---- Combine focal term with BCE ----
        loss = focal_weight * bce

        # ---- Reduction ----
        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss

freq = labels_df.mean()
inv_freq = 1.0/freq
alpha = (inv_freq/inv_freq.max()).to_numpy()

criterion = FocalLoss(
    gamma = 2.0, 
    alpha = alpha,
    reduction="mean"
)

In [28]:
model = MultiHeadEfficientNet(
    num_conditions=5, 
    hidden_dim=512, 
    dropout_rate=0.3
).to(device)

if has_checkpoint: 
    print("Loading model state dict from checkpoint")
    model.load_state_dict(checkpoint["model_state_dict"])

In [29]:
# initially, we are going to freeze the weights of the backbone and just train new features
if current_epoch < num_epochs_frozen:
    for param in model.backbone.parameters():
        param.requires_grad = False 
    
opt = torch.optim.AdamW(
    [
        {"params": model.backbone.parameters(), "lr": 0.0},
        {"params": model.shared_feature_processor.parameters(), "lr": 1.0e-3},
        {"params": model.heads.parameters(), "lr": 1.0e-3}
    ],
    weight_decay = 1.0e-3
)

if has_checkpoint: 
    print("Loading up optimizer state dict")
    opt.load_state_dict(checkpoint["opt_state_dict"])

In [30]:
def get_lr(
        current_epoch, 
        frozen_epochs=num_epochs_frozen, 
        warmup_epochs=3, 
        decay_epochs=80,
        total_epochs=num_epochs_total, 
        min_lrs=[1.0e-6, 5.0e-6, 1.0e-5],
        max_lrs=[1.0e-4, 1.0e-3, 1.0e-3],
        ):
    
    out_lrs = {"backbone": None, "shared": None, "head": None}
    
    if current_epoch < frozen_epochs: 
        out_lrs["backbone"] = 0.0 
        out_lrs["shared"] = max_lrs[1]
        out_lrs["head"] = max_lrs[2]
        return out_lrs 
    
    if current_epoch < frozen_epochs + warmup_epochs: 
        inv_warmup_epochs = current_epoch - frozen_epochs
        out_lrs["backbone"] = max_lrs[0] * (inv_warmup_epochs / warmup_epochs)
        out_lrs["shared"] = max_lrs[1]
        out_lrs["head"] = max_lrs[2] 
        return out_lrs 
    
    if current_epoch < frozen_epochs + warmup_epochs + decay_epochs: 
        decay_epochs = frozen_epochs + warmup_epochs + decay_epochs - frozen_epochs - warmup_epochs
        decay_progress = (current_epoch - frozen_epochs - warmup_epochs) / decay_epochs
        decay_progress = min(max(decay_progress, 0), 1)  # clamp
        out_lrs["backbone"] = min_lrs[0] + (max_lrs[0] - min_lrs[0]) * 0.5 * (1 + math.cos(math.pi * decay_progress))
        out_lrs["shared"] = min_lrs[1] + (max_lrs[1] - min_lrs[1]) * 0.5 * (1 + math.cos(math.pi * decay_progress))
        out_lrs["head"] = min_lrs[2] + (max_lrs[2] - min_lrs[2]) * 0.5 * (1 + math.cos(math.pi * decay_progress))
        return out_lrs
    
    out_lrs["backbone"] = min_lrs[0]
    out_lrs["shared"] = min_lrs[1]
    out_lrs["head"] = min_lrs[2]
    
    return out_lrs

In [31]:
class EarlyStopping:
    """
    Early stops the training if validation loss doesn't improve after 'patience' epochs.
    Saves the best model automatically.
    """
    def __init__(self, patience=3, verbose=True, delta=0.0, best_loss=None, save_path="best_checkpoint.pth"):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.save_path = save_path
        
        self.best_loss = best_loss if best_loss is not None else float("inf")
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss, epoch, model, opt, extra_state=None):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            
            # save best model
            checkpoint = {
                "model_state_dict": model.state_dict(),
                "opt_state_dict": opt.state_dict(),
                "epoch": epoch,
            }
            if extra_state:
                checkpoint.update(extra_state)
            torch.save(checkpoint, self.save_path)
            
            if self.verbose:
                print(f"  ✓ Validation improved → saving new best model (loss={val_loss:.5f})")
        else:
            self.counter += 1
            if self.verbose:
                print(f"  ✗ No improvement ({self.counter}/{self.patience})")
            if self.counter >= self.patience:
                self.early_stop = True


In [32]:
accumulation_steps = 1 #  Effective batch size = batch_size * accumulation_steps
scaler = GradScaler('cuda') if device.type == "cuda" else None
train_losses = []
train_epoch_stats = []
val_losses = []
val_ranking_stats = []

early_stopper = EarlyStopping(
    patience=3,
    verbose=True,
    save_path="best_model.pth",
    best_loss=None
)
label_smoothing_eta = 0.03

def label_smoothing(labels, eta=label_smoothing_eta): 
    return labels * (1-eta) + 0.5 * eta

for epoch in range(current_epoch, num_epochs_total): 
        
    if epoch == num_epochs_frozen: 
        # unfreeze parameters
        for param in model.backbone.parameters(): 
            param.requires_grad = True 
            
    # get learning rates for current epoch
    lrs = get_lr(epoch)
    opt.param_groups[0]["lr"] = lrs["backbone"]
    opt.param_groups[1]["lr"] = lrs["shared"]
    opt.param_groups[2]["lr"] = lrs["head"]
    print("="*100)
    print(f"For epoch {epoch}, using learning rates {lrs}")
    
    model.train()
    opt.zero_grad()
    pbar = tqdm(total=len(train_dataloader),
                desc=f"Epoch {epoch} - Training", 
                unit="batch")
    running_train_loss = torch.tensor(0.0, device=device)
    total_samples = 0
    train_batch_stats = []
    for i, (inputs, labels) in enumerate(train_dataloader): 
        
        if (i + 1) % 1 == 0 or (i + 1) == len(train_dataloader):
            pbar.n = i + 1
            pbar.refresh()
        
        inputs = inputs.to(device)
        labels = labels.to(device)
        labels_smooth = label_smoothing(labels)
        
        if device.type == 'cuda':
            with autocast('cuda'): 
                outputs = model(inputs)
                loss = criterion(outputs, labels_smooth)
                loss = loss / accumulation_steps
            scaler.scale(loss).backward()
        else: 
            # doesn't support mixed precision training
            outputs = model(inputs)
            loss = criterion(outputs, labels_smooth)
            loss = loss / accumulation_steps
            loss.backward()
            
        with torch.no_grad():
            y_pred = torch.sigmoid(outputs.float())     # convert logits → probabilities
            stats = batch_training_metrics(labels, y_pred)
            train_batch_stats.append(stats)
            
        if (i + 1) % accumulation_steps == 0:
            if device.type == "cuda":
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad()
            else: 
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                opt.step()
                opt.zero_grad()
                
        # accumulate metrics 
        running_train_loss = running_train_loss + (loss.detach() * accumulation_steps * inputs.shape[0])
        total_samples += inputs.shape[0]
    # in the edge case where num_batches is not divisible by accumulation steps, need to do one further step 
    if (i + 1) % accumulation_steps != 0: 
        if device.type == "cuda":
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad()
        else: 
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            opt.zero_grad()
        
    pbar.close()
        
    avg_train_loss = (running_train_loss / total_samples).item()
    train_losses.append({"epoch": epoch, "avg_train_loss": avg_train_loss})
    print(f"Epoch {epoch} Train Loss: ", avg_train_loss)
    train_stats_epoch = aggregate_training_epoch(train_batch_stats, total_samples)
    print(f"Epoch {epoch} TRAIN MONITOR:", train_stats_epoch)
    train_epoch_stats.append({"epoch": epoch, **train_stats_epoch})
    
    # VALIDATION LOOP   
    model.eval()
    pbar = tqdm(total=len(val_dataloader),
                desc=f"Epoch {epoch} - Validation", 
                unit="batch")
    running_val_loss = torch.tensor(0.0, device=device)
    total_samples = 0
    val_y_true_list = []
    val_y_pred_list = []
    with torch.no_grad(): 
        for i, (inputs, labels) in enumerate(val_dataloader): 
            if (i + 1) % 1 == 0 or (i + 1) == len(val_dataloader):
                pbar.n = i + 1
                pbar.refresh()

            inputs = inputs.to(device)
            labels = labels.to(device)
            
            if device.type == "cuda": 
                with autocast('cuda'): 
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
            y_pred = torch.sigmoid(outputs)
            val_y_pred_list.append(y_pred.cpu())
            val_y_true_list.append(labels.cpu())
            
            running_val_loss = running_val_loss + (loss * inputs.shape[0])
            total_samples += inputs.shape[0]
        
    pbar.close()
    
    all_y_true = torch.cat(val_y_true_list).numpy()
    all_y_pred = torch.cat(val_y_pred_list).numpy()
    
    avg_val_loss = (running_val_loss / total_samples).item()
    print(f"Epoch {epoch} Val Loss: ", avg_val_loss)
    val_losses.append({"epoch": epoch, "avg_val_loss": avg_val_loss})
    
    ranking_metrics = compute_ranking_metrics(all_y_true, all_y_pred)
    print(f"Epoch {epoch} VALIDATION RANKING:", ranking_metrics)
    val_ranking_stats.append({"epoch": epoch, **ranking_metrics})
    
    if epoch >= 10:
        print(f"Reached 10 epochs, outputting")
        checkpoint = {
            "model_state_dict": model.state_dict(), 
            "opt_state_dict": opt.state_dict(), 
            "epoch": epoch+1, 
            "lrs": lrs, 
            "val_losses": val_losses, 
            "train_losses": train_losses}
        torch.save(checkpoint, "checkpoint.pth")

        train_stats_df = pd.DataFrame(train_epoch_stats)
        train_stats_df.to_csv("train_epoch_stats.csv", index=False)
        train_losses_df = pd.DataFrame(train_losses)
        train_losses_df.to_csv("train_losses.csv", index=False)
        val_stats_df = pd.DataFrame(val_ranking_stats)
        val_stats_df.to_csv("val_ranking_stats.csv", index=False)
        val_losses_df = pd.DataFrame(val_losses)
        val_losses_df.to_csv("val_losses.csv", index=False)

For epoch 0, using learning rates {'backbone': 0.0, 'shared': 0.001, 'head': 0.001}


Epoch 0 - Training: 100%|██████████| 126/126 [09:58<00:00,  4.75s/batch]


Epoch 0 Train Loss:  0.08638082444667816
Epoch 0 TRAIN MONITOR: {'mean_pred_prob_per_label': [0.32356593012809753, 0.2898809313774109, 0.30100351572036743, 0.30473724007606506, 0.30054715275764465], 'mean_true_prob_per_label': [0.24632993340492249, 0.12565314769744873, 0.24956457316875458, 0.2159741222858429, 0.06817616522312164], 'mean_prob_mass': 1.5197346979897246, 'mean_cardinality': 1.5197346984643074, 'mean_pred_when_positive': [0.32995903491973877, 0.2950775623321533, 0.29981595277786255, 0.30556735396385193, 0.3070693016052246], 'mean_pred_when_negative': [0.32147637009620667, 0.28913408517837524, 0.30139848589897156, 0.3045085072517395, 0.3000698983669281], 'calibration_gap': [0.008482664823532104, 0.005943477153778076, -0.0015825331211090088, 0.0010588467121124268, 0.006999403238296509]}


Epoch 0 - Validation: 100%|██████████| 31/31 [02:21<00:00,  4.55s/batch]


Epoch 0 Val Loss:  0.0700351670384407
Epoch 0 VALIDATION RANKING: {'per_label_ap': [0.28262787679511436, 0.15260276915709794, 0.2811786507235674, 0.24887146814342703, 0.13390931724087643], 'per_label_auroc': [0.5746426476947593, 0.5602228799544972, 0.5156011934992382, 0.5367966162905289, 0.594296283038005], 'macro_ap': 0.21983801641201667, 'macro_auroc': 0.5563119240954058, 'micro_ap': 0.20607716805859264, 'micro_auroc': 0.567547295455244, 'lrap': 0.7464285714285724}
For epoch 1, using learning rates {'backbone': 0.0, 'shared': 0.001, 'head': 0.001}


Epoch 1 - Training:  33%|███▎      | 42/126 [03:17<06:35,  4.71s/batch]

KeyboardInterrupt: 